In [17]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from torch.nn.functional import pad
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm

# Step 2: Define Configuration
device = "cuda"
model_name_or_path = 'bert-base-multilingual-uncased'
tokenizer_name_or_path = 'bert-base-multilingual-uncased'
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,  # Correct task type for sequence classification
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",  # Changed to a more specific prompt
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128  # Increased max_length to accommodate longer texts
lr = 3e-2
num_epochs = 50
batch_size = 8

# Step 3: Load Dataset
dataset = load_dataset("ought/raft", dataset_name)
classes = [k.replace("_", " ") for k in dataset["train"].features["Label"].names]
dataset = dataset.map(lambda x: {"text_label": [classes[label] for label in x["Label"]]}, batched=True, num_proc=1)

# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
target_max_length = max([len(tokenizer(class_label)["input_ids"]) for class_label in classes])
print("Target max length:", target_max_length)

# Step 5: Preprocess Function
def preprocess_function(examples):
    batch_size = len(examples[text_column])
    inputs = [f"{text_column} : {x} Label : " for x in examples[text_column]]
    targets = [str(x) for x in examples[label_column]]
    model_inputs = tokenizer(inputs, truncation=True, max_length=max_length, padding="max_length")
    labels = tokenizer(targets, truncation=True, max_length=target_max_length, padding="max_length")
    labels_tensor = torch.tensor(labels["input_ids"])  # Convert the list to a tensor
    labels = torch.argmax(labels_tensor, dim=1)  # Get the index of the maximum value in each row
    model_inputs["labels"] = labels  # Assign the labels tensor directly
    return model_inputs

# Step 6: Apply Preprocessing
processed_datasets = dataset.map(preprocess_function, batched=True, num_proc=1, remove_columns=dataset["train"].column_names, load_from_cache_file=False, desc="Running tokenizer on dataset")
train_dataset = processed_datasets["train"]
eval_dataset = processed_datasets["test"]

# Step 7: DataLoaders
train_dataloader = DataLoader(train_dataset, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)
eval_dataloader = DataLoader(eval_dataset, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)

# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=len(classes))
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())

# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

# Step 10: Training Loop
model = model.to(device)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        if all(tensor.dim() == 0 for tensor in batch["labels"]):  # Check if all tensors are 0-dimensional
            batch["labels"] = batch["labels"]  # Use the original list of 0-dimensional tensors
        else:
            max_target_length = max(tensor.size(0) for tensor in batch["labels"] if tensor.dim() > 0)  # Get the max length of the tensors
            padded_targets = [pad(tensor, (0, max_target_length - tensor.size(0))) if tensor.dim() > 0 else tensor.unsqueeze(0) for tensor in batch["labels"]]
            batch["labels"] = torch.stack(padded_targets)
        print("Input shape:", batch["input_ids"].shape)
        print("Target shape:", batch["labels"].shape)
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"Epoch {epoch}: train_ppl={train_ppl}, train_loss={train_epoch_loss}, eval_ppl={eval_ppl}, eval_loss={eval_epoch_loss}")


Target max length: 6


Running tokenizer on dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/3399 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 8,451 || all params: 167,367,174 || trainable%: 0.0050
None


...

Input shape: torch.Size([8, 128])
Target shape: torch.Size([8])
Input shape: torch.Size([8, 128])
Target shape: torch.Size([8])


100%|██████████| 7/7 [00:00<00:00,  8.67it/s]


Input shape: torch.Size([2, 128])
Target shape: torch.Size([2])


100%|██████████| 425/425 [00:27<00:00, 15.22it/s]

Epoch 49: train_ppl=1.375073790550232, train_loss=0.3185074031352997, eval_ppl=1.5159293413162231, eval_loss=0.4160286486148834


In [18]:
inputs = tokenizer(
    f'{text_column} : {"@nationalgridus I have no water and the bill is current and paid. Can you do something about this?"} Label : ',
    return_tensors="pt",
)

In [19]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(
        input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], max_new_tokens=10, eos_token_id=3
    )
    print(tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True))

TypeError: The current model class (BertForSequenceClassification) is not compatible with `.generate()`, as it doesn't have a language model head. Please use one of the following classes instead: {'BertLMHeadModel'}

In [ ]:
torch.save(model.state_dict(), 'bert_model.pth')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-multilingual-uncased', num_labels=len(classes))
model.load_state_dict(torch.load('bert_model.pth'))
model.eval()

In [ ]:
inputs = tokenizer(
    f'{text_column} : {"@nationalgridus The Geyser is not working at all. Can you do something about this?"} Label : ',
    return_tensors="pt",
)

In [ ]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(
        input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], max_new_tokens=10, eos_token_id=3
    )
    print(outputs)
    print(tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True))